# Bias Kebocoran Data akibat SMOTE sebelum Pembagian Data pada Prediksi Stroke
**Re-evaluasi dengan Nested Cross-Validation dan Kalibrasi** — notebook lengkap untuk naskah JIEE

Isi notebook:
- **Bagian A** — Skrining literatur Scopus (81 artikel inti, risiko kebocoran, Excel)
- **Bagian B** — Eksperimen 1: skenario bocor vs benar + mekanisme kebocoran
- **Bagian C** — Eksperimen 2: 4 strategi ketidakseimbangan × model, default vs tuned (nested CV), uji statistik
- **Bagian D** — Eksperimen 3: model terbaik → kalibrasi isotonik, ambang Youden, permutation importance
- **Bagian E** — Gambar dan tabel klaim literatur vs hasil bebas kebocoran

Cukup ubah `DATA_DIR` di sel konfigurasi, lalu jalankan *Run All*. Waktu jalan penuh ±20–30 menit (lebih lama bila Optuna/XGBoost/LightGBM aktif). Set `FAST = True` untuk uji cepat ±3 menit.

## 0. Instalasi (opsional)
Jalankan sekali bila paket belum ada. Optuna, XGBoost, dan LightGBM **opsional**: aktifkan lewat `USE_OPTUNA` dan `USE_EXTRA_MODELS` di sel konfigurasi.

In [ ]:
# !pip install numpy pandas scikit-learn scipy matplotlib openpyxl
# !pip install optuna xgboost lightgbm      # opsional

## 1. Konfigurasi

In [ ]:
import os, re, glob, json, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

# ==== Struktur repositori: data di ../data, hasil di ../results ====
DATA_DIR = os.path.join("..", "data")
STROKE_CSV = os.path.join(DATA_DIR, "healthcare-dataset-stroke-data.csv")
SCOPUS_GLOB = os.path.join(DATA_DIR, "export_*.csv")       # file ekspor Scopus
OUT = os.path.join("..", "results")
os.makedirs(OUT, exist_ok=True)

SEED = 42
FAST = False          # True = uji cepat (1 pengulangan, 4 kandidat tuning)
USE_OPTUNA = False        # True = tuning dengan Optuna TPE (perlu paket optuna)
USE_EXTRA_MODELS = False  # True = tambah XGBoost & LightGBM (perlu paketnya)
# Default False/False = mereproduksi angka di naskah (RandomizedSearchCV; LR, RF, HGB)
N_REPEATS = 1 if FAST else 2      # repeated stratified 5-fold (outer)
N_ITER    = 4 if FAST else 12     # jumlah kandidat hyperparameter per fold
N_BOOT    = 300 if FAST else 1000

try:
    import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING); HAS_OPTUNA = USE_OPTUNA
except ImportError:
    HAS_OPTUNA = False
print("Optuna aktif:", HAS_OPTUNA, "| model tambahan:", USE_EXTRA_MODELS, "| FAST:", FAST, "| output:", OUT)

# Bagian A — Skrining literatur Scopus
Menggabungkan ekspor Scopus, deduplikasi EID, menyaring artikel yang memakai dataset stroke Kaggle (tabular, bukan citra), mengekstrak klaim akurasi, dan memberi label risiko kebocoran dari abstrak. **Hasil skrining otomatis wajib dicek manual dengan teks lengkap.**

In [ ]:
files = sorted(glob.glob(SCOPUS_GLOB))
if not files:
    print("File Scopus tidak ditemukan — Bagian A dilewati.")
    u = None
else:
    d = pd.concat([pd.read_csv(f, low_memory=False) for f in files])
    print("Total baris:", len(d))
    d["key"] = d.EID.fillna(d.Title.str.lower())
    u = d.drop_duplicates("key").copy()
    print("Dokumen unik:", len(u))
    u["txt"] = (u.Title.fillna("") + " " + u.Abstract.fillna("") + " " + u["Author Keywords"].fillna("")).str.lower()
    display(u.Year.value_counts().sort_index().to_frame("jumlah"))

In [ ]:
PATS = {
 "stroke": r"stroke", "ml": r"machine learning|deep learning|classifier|random forest|xgboost", "predict": r"predict",
 "imaging": r"\bct\b|mri|imaging|computed tomography",
 "oversampling": r"smote|oversampl|adasyn|imbalanc", "tuning": r"hyperparamet|grid search|random search|bayesian optim|optuna|tuning",
 "optuna": r"optuna", "xai": r"shap|lime\b|explainab|interpretab", "calibration": r"calibrat", "leakage": r"leakage|leak",
 "nested_cv": r"nested cross", "threshold": r"threshold", "pr_auc": r"pr-auc|precision-recall curve|auprc|average precision",
 "mcc": r"matthews|\bmcc\b", "brier": r"brier",
}
if u is not None:
    for k, p in PATS.items(): u[k] = u.txt.str.contains(p, regex=True)
    print("stroke + ML + prediksi:", (u.stroke & u.ml & u.predict).sum(), "| berbasis citra:", (u.stroke & u.ml & u.predict & u.imaging).sum())
    t = u.Title.fillna("").str.lower()
    k = u[u.txt.str.contains(r"kaggle|healthcare-dataset|fedesoriano|5,?110 (?:patients|records|instances|samples)|stroke prediction dataset")
          & t.str.contains("stroke") & ~t.str.contains(r"image|ct |mri|segment|eeg|tumor|lesion")].copy()
    print("Artikel inti (dataset stroke Kaggle, tabular):", len(k))
    display(k.Year.value_counts().sort_index().to_frame("jumlah"))

In [ ]:
def max_accuracy(abstract):
    t = str(abstract).lower()
    v = [float(x) for x in re.findall(r"accura\w*[^.;]{0,60}?(\d{2,3}(?:\.\d+)?)\s*%", t)]
    v += [float(x) * 100 for x in re.findall(r"accura\w*[^;]{0,40}?\b(0\.\d{2,4})\b", t)]
    v = [x for x in v if 50 <= x <= 100]
    return max(v) if v else np.nan

MODELS_RX = [("Random Forest", r"random forest"), ("XGBoost", r"xgboost|xgb\b"), ("LightGBM", r"lightgbm|lgbm"), ("CatBoost", r"catboost"),
             ("SVM", r"\bsvm\b|support vector"), ("KNN", r"\bknn\b|k-nearest|nearest neighbo"), ("Logistic Regression", r"logistic regression"),
             ("Decision Tree", r"decision tree"), ("Naive Bayes", r"naive bayes|naïve bayes"), ("Neural Network/DL", r"neural network|deep learning|\bann\b|\bdnn\b|lstm|cnn"),
             ("Stacking/Ensemble", r"stacking|ensemble|voting"), ("AdaBoost/GBM", r"adaboost|gradient boosting")]

def imbalance_method(s):
    o = []
    if re.search(r"smote-?enn|smoteenn", s): o.append("SMOTE-ENN")
    elif re.search(r"smote-?tomek", s): o.append("SMOTE-Tomek")
    elif re.search(r"borderline", s): o.append("Borderline-SMOTE")
    elif re.search(r"smote", s): o.append("SMOTE")
    if re.search(r"adasyn", s): o.append("ADASYN")
    if re.search(r"random over|oversampl", s) and not o: o.append("Oversampling")
    if re.search(r"undersampl|under-sampl|near ?miss", s): o.append("Undersampling")
    if re.search(r"class weight|cost-sensitive|weighted", s): o.append("Class weight")
    return ", ".join(o)

RX_CORRECT = r"only (on|to) the training|training (set|data) only|after (the )?(train|data) split|after splitting|within (each )?(training )?fold|inside (the )?cross-validation|without data leakage|avoid(ing)? (data )?leakage|prevent(ing)? (data )?leakage"
RX_LEAKY = r"(smote|oversampl|balanc)[^.]{0,80}(entire|whole|full) dataset|before (splitting|the split|train)"

def leakage_risk(r):
    over = any(x in r.imb for x in ["SMOTE", "ADASYN", "Oversampling"])
    if re.search(RX_LEAKY, r.txt): return "Tinggi", "Abstrak menyebut resampling sebelum split / pada seluruh data"
    if re.search(RX_CORRECT, r.txt): return "Rendah", "Abstrak menyebut resampling hanya di data latih / tanpa kebocoran"
    if over and r.acc >= 95: return "Tinggi", "Oversampling + akurasi ≥95% pada data prevalensi 4,9%"
    if r.acc >= 95: return "Sedang", "Akurasi ≥95% (bisa paradoks akurasi atau kebocoran)"
    if over: return "Sedang", "Oversampling dipakai, urutan terhadap split tidak disebut"
    if pd.notna(r.acc): return "Rendah", "Akurasi <95%, tidak ada indikasi di abstrak"
    return "Tidak dapat dinilai", "Abstrak tidak memuat cukup informasi"

if u is not None:
    k["acc"] = k.Abstract.apply(max_accuracy)
    k["models"] = k.txt.apply(lambda s: ", ".join(n for n, p in MODELS_RX if re.search(p, s)))
    k["imb"] = k.txt.apply(imbalance_method)
    k[["risk", "reason"]] = k.apply(lambda r: pd.Series(leakage_risk(r)), axis=1)
    a = k.acc
    print(f"Menyebut akurasi: {a.notna().sum()} | ≥95%: {(a>=95).sum()} | ≥99%: {(a>=99).sum()} | median: {a.median():.1f}%")
    for c in ["oversampling", "tuning", "optuna", "nested_cv", "calibration", "brier", "leakage", "pr_auc", "mcc", "xai"]:
        print(f"  {c:12s} {k[c].sum()}")
    display(k.risk.value_counts().to_frame("jumlah"))

In [ ]:
if u is not None:
    order = {"Tinggi": 0, "Sedang": 1, "Rendah": 2, "Tidak dapat dinilai": 3}
    k = k.sort_values(["risk", "Year"], key=lambda c: c.map(order) if c.name == "risk" else -c).reset_index(drop=True)
    screen = pd.DataFrame({
        "No": range(1, len(k) + 1), "Penulis pertama": k.Authors.fillna("").str.split(";").str[0].str.strip(), "Authors": k.Authors,
        "Tahun": k.Year, "Judul": k.Title, "Sumber": k["Source title"], "Tipe": k["Document Type"], "Sitasi": k["Cited by"], "DOI": k.DOI,
        "Model disebut": k.models, "Penanganan imbalance": k.imb, "Akurasi maks di abstrak (%)": k.acc,
        "Tuning hyperparameter": k.tuning.map({True: "Ya", False: ""}), "XAI/SHAP": k.xai.map({True: "Ya", False: ""}),
        "Risiko kebocoran (skrining otomatis)": k.risk, "Alasan skrining": k.reason,
        "CEK MANUAL: urutan resampling vs split (full text)": "", "CEK MANUAL: metrik dilaporkan": "", "CEK MANUAL: catatan": "",
        "Abstrak": k.Abstract})
    display(screen.head(10))

# Bagian B — Eksperimen 1: skenario bocor vs benar

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import (StratifiedKFold, RepeatedStratifiedKFold, RandomizedSearchCV,
                                     train_test_split, cross_val_predict, cross_val_score)
from sklearn.metrics import (accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, recall_score,
                             precision_score, f1_score, brier_score_loss, roc_curve, confusion_matrix, precision_recall_curve)
from sklearn.isotonic import IsotonicRegression
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
from scipy.stats import wilcoxon, friedmanchisquare
import matplotlib.pyplot as plt

df = pd.read_csv(STROKE_CSV).drop(columns="id")
df = df[df.gender != "Other"].reset_index(drop=True)
y = df.pop("stroke").values
X = df
NUM = ["age", "avg_glucose_level", "bmi"]; BIN = ["hypertension", "heart_disease"]
CAT = ["gender", "ever_married", "work_type", "Residence_type", "smoking_status"]
print("Data:", X.shape, "| stroke:", y.sum(), f"({y.mean():.2%}) | BMI kosong:", X.bmi.isna().sum())

def make_prep():
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), NUM),
        ("bin", "passthrough", BIN),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False), CAT)])

In [ ]:
def smote(X, y, k=5, seed=0):
    # SMOTE (Chawla et al., 2002): interpolasi linear antar tetangga kelas minoritas
    r = np.random.RandomState(seed); Xm = X[y == 1]; n = int((y == 0).sum() - len(Xm))
    if n <= 0: return X, y
    nn = NearestNeighbors(n_neighbors=min(k + 1, len(Xm))).fit(Xm)
    idx = nn.kneighbors(Xm, return_distance=False)[:, 1:]
    i = r.randint(0, len(Xm), n); j = idx[i, r.randint(0, idx.shape[1], n)]; g = r.rand(n, 1)
    return np.vstack([X, Xm[i] + g * (Xm[j] - Xm[i])]), np.r_[y, np.ones(n, int)]

def rus(X, y, seed=0):
    r = np.random.RandomState(seed); pos = np.where(y == 1)[0]; neg = np.where(y == 0)[0]
    keep = np.r_[pos, r.choice(neg, len(pos), replace=False)]
    return X[keep], y[keep]

class Resampled(ClassifierMixin, BaseEstimator):
    # Resampling HANYA saat fit (data latih). Urutan pewarisan (ClassifierMixin dulu) penting
    # agar scorer scikit-learn mengenali ini sebagai classifier.
    def __init__(self, estimator=None, sampler="none", random_state=0):
        self.estimator = estimator; self.sampler = sampler; self.random_state = random_state
    def fit(self, X, y):
        X = np.asarray(X, float); y = np.asarray(y)
        if self.sampler == "smote": X, y = smote(X, y, seed=self.random_state)
        elif self.sampler == "rus": X, y = rus(X, y, seed=self.random_state)
        self.est_ = clone(self.estimator).fit(X, y); self.classes_ = self.est_.classes_; return self
    def predict_proba(self, X): return self.est_.predict_proba(np.asarray(X, float))
    def predict(self, X): return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)

SPW = float((y == 0).sum() / (y == 1).sum())   # untuk XGBoost scale_pos_weight

def models(cw=False):
    w = "balanced" if cw else None
    m = {
        "LR": (LogisticRegression(max_iter=3000, class_weight=w), {"C": list(np.logspace(-3, 2, 30))}),
        "RF": (RandomForestClassifier(n_estimators=200, class_weight=w, n_jobs=-1, random_state=SEED),
               {"max_depth": [3, 5, 8, 12, None], "min_samples_leaf": [1, 5, 10, 20, 50], "max_features": ["sqrt", 0.5, 0.8]}),
        "HGB": (HistGradientBoostingClassifier(class_weight=w, random_state=SEED),
                {"learning_rate": list(np.logspace(-2.5, -0.5, 20)), "max_depth": [2, 3, 4, None], "max_leaf_nodes": [7, 15, 31],
                 "min_samples_leaf": [20, 50, 100], "l2_regularization": list(np.logspace(-3, 1, 10))}),
    }
    if not USE_EXTRA_MODELS: return m
    try:
        from xgboost import XGBClassifier
        m["XGB"] = (XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=SEED, n_jobs=-1, scale_pos_weight=(SPW if cw else 1.0)),
                    {"max_depth": [2, 3, 4, 6], "learning_rate": list(np.logspace(-2.5, -0.5, 20)), "subsample": [0.6, 0.8, 1.0],
                     "colsample_bytree": [0.6, 0.8, 1.0], "min_child_weight": [1, 5, 10]})
    except ImportError: pass
    try:
        from lightgbm import LGBMClassifier
        m["LGBM"] = (LGBMClassifier(n_estimators=300, class_weight=w, random_state=SEED, verbose=-1),
                     {"num_leaves": [7, 15, 31], "learning_rate": list(np.logspace(-2.5, -0.5, 20)),
                      "min_child_samples": [10, 20, 50, 100], "reg_lambda": list(np.logspace(-3, 1, 10))})
    except ImportError: pass
    return m

def metrics(yt, p, thr=0.5):
    yh = (p >= thr).astype(int); tn, fp, fn, tp = confusion_matrix(yt, yh, labels=[0, 1]).ravel()
    return dict(Accuracy=accuracy_score(yt, yh), ROC_AUC=roc_auc_score(yt, p), PR_AUC=average_precision_score(yt, p),
                Recall=recall_score(yt, yh, zero_division=0), Specificity=tn / (tn + fp), Precision=precision_score(yt, yh, zero_division=0),
                F1=f1_score(yt, yh, zero_division=0), MCC=matthews_corrcoef(yt, yh), Brier=brier_score_loss(yt, p))

def boot_ci(yt, p, fn, B=N_BOOT, seed=SEED):
    r = np.random.RandomState(seed); v = []
    for _ in range(B):
        i = r.randint(0, len(yt), len(yt))
        if yt[i].sum() > 0: v.append(fn(yt[i], p[i]))
    return np.percentile(v, [2.5, 97.5])

print("Model tersedia:", list(models().keys()))

In [ ]:
t0 = time.time()
outer = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_REPEATS, random_state=SEED)
inner = StratifiedKFold(3, shuffle=True, random_state=SEED)
allm = models(False); allm["KNN"] = (KNeighborsClassifier(5), {})
rows1 = []

# A: praproses + SMOTE pada SELURUH data, lalu split 80:20  (pola umum di literatur — BOCOR)
Xp = make_prep().fit_transform(X); Xs, ys = smote(Xp, y, seed=SEED)
xtr, xte, ytr, yte = train_test_split(Xs, ys, test_size=0.2, stratify=ys, random_state=SEED)
for name, (est, _) in allm.items():
    rows1.append(dict(Skenario="A. Bocor: SMOTE sebelum split (80/20)", Model=name, **metrics(yte, clone(est).fit(xtr, ytr).predict_proba(xte)[:, 1])))
# A2: SMOTE seluruh data lalu 5-fold CV (BOCOR)
for name, (est, _) in allm.items():
    p = cross_val_predict(clone(est), Xs, ys, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method="predict_proba")[:, 1]
    rows1.append(dict(Skenario="A2. Bocor: SMOTE sebelum 5-fold CV", Model=name, **metrics(ys, p)))
# B: split dulu, praproses + SMOTE hanya di data latih (BENAR)
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
for name, (est, _) in allm.items():
    pipe = Pipeline([("prep", make_prep()), ("clf", Resampled(est, "smote", SEED))]).fit(xtr, ytr)
    rows1.append(dict(Skenario="B. Benar: split dulu, SMOTE hanya data latih (80/20)", Model=name, **metrics(yte, pipe.predict_proba(xte)[:, 1])))
# B2: SMOTE di dalam fold latih, repeated CV (BENAR)
for name, (est, _) in allm.items():
    pipe = Pipeline([("prep", make_prep()), ("clf", Resampled(est, "smote", SEED))])
    P = np.zeros(len(y)); cnt = np.zeros(len(y))
    for tr, te in outer.split(X, y):
        P[te] += clone(pipe).fit(X.iloc[tr], y[tr]).predict_proba(X.iloc[te])[:, 1]; cnt[te] += 1
    rows1.append(dict(Skenario="B2. Benar: SMOTE di dalam fold latih (CV)", Model=name, **metrics(y, P / cnt)))

r1 = pd.DataFrame(rows1); r1.to_csv(f"{OUT}/eksp1_leaky_vs_benar.csv", index=False)
print(f"selesai {time.time()-t0:.0f} detik"); display(r1.round(3))

In [ ]:
# Mekanisme kebocoran: berapa kasus positif di data uji skenario A yang sintetis, dan seberapa dekat ke data latih?
Xp = make_prep().fit_transform(X); Xs, ys = smote(Xp, y, seed=SEED)
syn = np.r_[np.zeros(len(y), bool), np.ones(len(ys) - len(y), bool)]
itr, ite = train_test_split(np.arange(len(ys)), test_size=0.2, stratify=ys, random_state=SEED)
pos_te = ite[ys[ite] == 1]
d_syn = NearestNeighbors(n_neighbors=1).fit(Xs[itr][ys[itr] == 1]).kneighbors(Xs[pos_te][syn[pos_te]])[0].ravel()
d_real = NearestNeighbors(n_neighbors=2).fit(Xp[y == 1]).kneighbors(Xp[y == 1])[0][:, 1]
leak = dict(n_test_positif=int(len(pos_te)), n_test_positif_sintetis=int(syn[pos_te].sum()),
            persen_sintetis=round(100 * syn[pos_te].mean(), 1),
            median_jarak_sintetis_uji_ke_latih=float(np.median(d_syn)), median_jarak_antar_pasien_stroke_asli=float(np.median(d_real)))
leak["rasio_jarak"] = leak["median_jarak_antar_pasien_stroke_asli"] / leak["median_jarak_sintetis_uji_ke_latih"]
json.dump(leak, open(f"{OUT}/mekanisme_kebocoran.json", "w"), indent=1); leak

# Bagian C — Eksperimen 2: strategi ketidakseimbangan × model, default vs tuned (nested CV)
Outer: repeated stratified 5-fold (2 pengulangan = 10 fold). Inner: stratified 3-fold untuk pemilihan hyperparameter (objektif PR-AUC). Bila `USE_OPTUNA = True` dan paketnya terpasang, pencarian memakai Optuna TPE; jika tidak, `RandomizedSearchCV` (seperti di naskah).

In [ ]:
def tune(pipe, space, Xtr, ytr, n_iter=N_ITER, seed=SEED):
    # Kembalikan pipeline terbaik yang sudah di-fit pada Xtr (pemilihan parameter hanya memakai Xtr)
    sp = {f"clf__estimator__{k}": v for k, v in space.items()}
    if HAS_OPTUNA:
        def objective(trial):
            params = {}
            for k, v in sp.items():
                choice = trial.suggest_categorical(k, list(range(len(v))))   # indeks agar aman untuk None/float
                params[k] = v[choice]
            return cross_val_score(clone(pipe).set_params(**params), Xtr, ytr, cv=inner, scoring="average_precision", error_score="raise").mean()
        study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_iter, show_progress_bar=False)
        best = {k: sp[k][study.best_params[k]] for k in sp}
        return clone(pipe).set_params(**best).fit(Xtr, ytr), best
    rs = RandomizedSearchCV(pipe, sp, n_iter=n_iter, scoring="average_precision", cv=inner,
                            random_state=seed, n_jobs=1, error_score="raise").fit(Xtr, ytr)
    return rs.best_estimator_, rs.best_params_

# uji cepat: scorer harus bekerja (mencegah tuning diam-diam gagal)
_p = Pipeline([("prep", make_prep()), ("clf", Resampled(models(True)["LR"][0], "none"))]).fit(X, y)
from sklearn.metrics import get_scorer
assert abs(get_scorer("average_precision")(_p, X, y) - average_precision_score(y, _p.predict_proba(X)[:, 1])) < 1e-9
print("Scorer OK")

In [ ]:
t0 = time.time()
rows2, fold_rows, preds = [], [], {}
for strat in ["none", "smote", "rus", "class_weight"]:
    samp = strat if strat in ("smote", "rus") else "none"
    for name, (est, space) in models(cw=(strat == "class_weight")).items():
        for mode in ["default", "tuned"]:
            P = np.zeros(len(y)); cnt = np.zeros(len(y))
            for f, (tr, te) in enumerate(outer.split(X, y)):
                pipe = Pipeline([("prep", make_prep()), ("clf", Resampled(est, samp, SEED))])
                if mode == "tuned": pipe, _ = tune(pipe, space, X.iloc[tr], y[tr], seed=SEED + f)
                else: pipe.fit(X.iloc[tr], y[tr])
                p = pipe.predict_proba(X.iloc[te])[:, 1]; P[te] += p; cnt[te] += 1
                fold_rows.append(dict(Strategi=strat, Model=name, Mode=mode, Fold=f, PR_AUC=average_precision_score(y[te], p),
                                      ROC_AUC=roc_auc_score(y[te], p), MCC=matthews_corrcoef(y[te], (p >= .5).astype(int))))
            P /= cnt; preds[(strat, name, mode)] = P; m = metrics(y, P)
            lo, hi = boot_ci(y, P, average_precision_score); lo2, hi2 = boot_ci(y, P, roc_auc_score)
            rows2.append(dict(Strategi=strat, Model=name, Mode=mode, **m, PR_AUC_CI=f"{lo:.3f}-{hi:.3f}", ROC_AUC_CI=f"{lo2:.3f}-{hi2:.3f}"))
            print(f"[{time.time()-t0:5.0f}s] {strat:12s} {name:4s} {mode:7s} ROC={m['ROC_AUC']:.3f} PR={m['PR_AUC']:.3f} MCC={m['MCC']:.3f} Brier={m['Brier']:.3f}")
r2 = pd.DataFrame(rows2); r2.to_csv(f"{OUT}/eksp2_nested_cv.csv", index=False)
fr = pd.DataFrame(fold_rows); fr.to_csv(f"{OUT}/eksp2_per_fold.csv", index=False)
display(r2.round(3))

In [ ]:
# Uji statistik: Wilcoxon (default vs tuned, per strategi-model) dan Friedman (antar strategi, model tuned)
# Catatan: fold dari repeated CV tidak sepenuhnya independen -> tafsirkan nilai p secara hati-hati
st = []
for (s, mdl), g in fr.groupby(["Strategi", "Model"]):
    a = g[g.Mode == "default"].sort_values("Fold").PR_AUC.values; b = g[g.Mode == "tuned"].sort_values("Fold").PR_AUC.values
    try: pv = wilcoxon(a, b).pvalue
    except ValueError: pv = np.nan
    st.append(dict(Uji="Wilcoxon default vs tuned (PR-AUC)", Strategi=s, Model=mdl, Delta=(b - a).mean(), p=pv))
for mdl, g in fr[fr.Mode == "tuned"].groupby("Model"):
    piv = g.pivot(index="Fold", columns="Strategi", values="PR_AUC")
    st.append(dict(Uji="Friedman antar strategi (tuned, PR-AUC)", Strategi="|".join(piv.columns), Model=mdl, Delta=np.nan,
                   p=friedmanchisquare(*[piv[c] for c in piv.columns]).pvalue))
st = pd.DataFrame(st); st.to_csv(f"{OUT}/eksp2_uji_statistik.csv", index=False); display(st.round(4))

# Bagian D — Eksperimen 3: kalibrasi, ambang keputusan, dan kontribusi fitur
Model tuned dengan PR-AUC tertinggi dilatih ulang pada 80% data. Kalibrasi isotonik dan ambang Youden **hanya** dipelajari dari prediksi out-of-fold data latih.

In [ ]:
best = r2[r2.Mode == "tuned"].sort_values("PR_AUC", ascending=False).iloc[0]
strat, name = best.Strategi, best.Model; print("Model terbaik:", strat, name)
est, space = models(cw=(strat == "class_weight"))[name]; samp = strat if strat in ("smote", "rus") else "none"
xtr, xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
raw, best_params = tune(Pipeline([("prep", make_prep()), ("clf", Resampled(est, samp, SEED))]), space, xtr, ytr, n_iter=N_ITER * 2)
print("Parameter terbaik:", best_params)

cv5 = StratifiedKFold(5, shuffle=True, random_state=SEED)
oof_raw = cross_val_predict(clone(raw), xtr, ytr, cv=cv5, method="predict_proba")[:, 1]
iso = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(oof_raw, ytr)
p_raw = raw.predict_proba(xte)[:, 1]; p_cal = iso.predict(p_raw)
rows3 = []
for lab, oof, p in [("Tanpa kalibrasi", oof_raw, p_raw), ("Kalibrasi isotonik", iso.predict(oof_raw), p_cal)]:
    fpr, tpr, th = roc_curve(ytr, oof); thr = float(th[np.argmax(tpr - fpr)])
    rows3.append(dict(Versi=lab, Ambang="0.5", **metrics(yte, p, 0.5)))
    rows3.append(dict(Versi=lab, Ambang=f"Youden OOF latih = {thr:.3f}", **metrics(yte, p, thr)))
r3 = pd.DataFrame(rows3); r3.to_csv(f"{OUT}/eksp3_kalibrasi_ambang.csv", index=False)
json.dump({k: str(v) for k, v in best_params.items()}, open(f"{OUT}/eksp3_best_params.json", "w"), indent=1)
display(r3.round(3))

In [ ]:
pi = permutation_importance(raw, xte, yte, scoring="average_precision", n_repeats=20, random_state=SEED)
imp = pd.DataFrame(dict(Fitur=X.columns, Importance_mean=pi.importances_mean, Importance_sd=pi.importances_std)).sort_values("Importance_mean", ascending=False)
imp.to_csv(f"{OUT}/eksp3_permutation_importance.csv", index=False); display(imp.round(4))

# Bagian E — Gambar dan tabel klaim literatur

In [ ]:
fc = lambda v: f"{v:.3f}".replace(".", ",")
fig, ax = plt.subplots(figsize=(5.2, 5))
for lab, p, col in [("Tanpa kalibrasi", p_raw, "#8c8c8c"), ("Kalibrasi isotonik", p_cal, "#1f5fa8")]:
    a, b = calibration_curve(yte, p, n_bins=8, strategy="quantile")
    ax.plot(b, a, "o-", color=col, label=f"{lab} (Brier {fc(brier_score_loss(yte, p))})")
ax.plot([0, 1], [0, 1], "k--", lw=.8, label="Kalibrasi sempurna")
ax.set_xlabel("Probabilitas stroke hasil prediksi"); ax.set_ylabel("Proporsi stroke teramati")
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.legend(loc="upper left", fontsize=8.5); fig.tight_layout()
fig.savefig(f"{OUT}/gambar_kalibrasi.png", dpi=300); plt.show()

fig, ax = plt.subplots(figsize=(5.8, 4.6))
for (s, mdl, mode), P in preds.items():
    if mode == "tuned" and s in ("smote", "class_weight"):
        pr, rc, _ = precision_recall_curve(y, P); ax.plot(rc, pr, label=f"{mdl} + {s} (AP {fc(average_precision_score(y, P))})")
ax.axhline(y.mean(), color="k", ls="--", lw=.8, label=f"Prevalensi {fc(y.mean())}")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision"); ax.legend(fontsize=7); fig.tight_layout()
fig.savefig(f"{OUT}/gambar_pr_curve.png", dpi=300); plt.show()

In [ ]:
# Tabel klaim literatur vs replikasi bocor vs hasil bebas kebocoran (per keluarga model)
if u is not None:
    def family(m):
        for key, f in [("Random Forest", "RF"), ("XGBoost", "HGB"), ("LightGBM", "HGB"), ("CatBoost", "HGB"), ("AdaBoost/GBM", "HGB"),
                       ("Stacking/Ensemble", "RF"), ("KNN", "KNN"), ("Logistic Regression", "LR")]:
            if key in (m or ""): return f, ""
        return "RF", " (model tidak disebut di abstrak)"
    A = r1[r1.Skenario.str.startswith("A. ")].set_index("Model"); B2 = r1[r1.Skenario.str.startswith("B2")].set_index("Model")
    c = screen[screen["Akurasi maks di abstrak (%)"].notna()].copy()
    fam = c["Model disebut"].apply(family); c["fam"] = fam.str[0]
    nm = {"RF": "Random Forest", "HGB": "Gradient boosting (HGB)", "KNN": "KNN", "LR": "Regresi logistik"}
    c["Model pembanding (eksperimen ini)"] = c.fam.map(nm) + fam.str[1]
    c["Replikasi bocor: Akurasi (%)"] = c.fam.map(A.Accuracy * 100).round(1)
    c["Bebas bocor: Akurasi (%)"] = c.fam.map(B2.Accuracy * 100).round(1)
    c["Bebas bocor: Recall"] = c.fam.map(B2.Recall).round(2)
    c["Bebas bocor: PR-AUC"] = c.fam.map(B2.PR_AUC).round(3)
    c["Bebas bocor: MCC"] = c.fam.map(B2.MCC).round(2)
    claims = c.drop(columns=["fam", "Abstrak", "Authors", "Tipe", "Sitasi"] + [x for x in c.columns if x.startswith("CEK")]) \
              .sort_values("Akurasi maks di abstrak (%)", ascending=False).reset_index(drop=True)
    a = claims["Akurasi maks di abstrak (%)"]
    print(f"Klaim ≥99%: {(a>=99).sum()} | 95–99%: {((a>=95)&(a<99)).sum()} | <95%: {(a<95).sum()} | replikasi bocor tertinggi: {A.Accuracy.max()*100:.1f}%")
    display(claims.head(15))

In [ ]:
# Simpan Excel skrining (3 sheet) dengan pewarnaan risiko dan kolom cek manual
if u is not None:
    from openpyxl import load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.worksheet.datavalidation import DataValidation
    xl = os.path.join(OUT, "81_artikel_inti_stroke_kaggle.xlsx")
    with pd.ExcelWriter(xl, engine="openpyxl") as w:
        screen.to_excel(w, sheet_name="Skrining artikel", index=False)
        claims.to_excel(w, sheet_name="Klaim vs bebas bocor", index=False)
        pd.DataFrame({"Kolom": ["Risiko kebocoran", "Tinggi", "Sedang", "Rendah", "Tidak dapat dinilai", "CEK MANUAL"],
                      "Keterangan": ["Aturan kata kunci pada abstrak; WAJIB diverifikasi dengan full text",
                                     "Resampling pada seluruh data/sebelum split, ATAU oversampling + akurasi ≥95%",
                                     "Akurasi ≥95% tanpa info urutan, atau oversampling tanpa info urutan split",
                                     "Resampling hanya di data latih, atau akurasi <95% tanpa indikasi",
                                     "Abstrak tidak memuat akurasi maupun teknik resampling",
                                     "Isi setelah membaca bagian Metode: Bocor / Benar / Tidak jelas"]}).to_excel(w, sheet_name="Keterangan", index=False)
    wb = load_workbook(xl); ws = wb["Skrining artikel"]; cols = {c.value: c.column_letter for c in ws[1]}
    for c in ws[1]: c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="1F4E78"); c.alignment = Alignment(wrap_text=True)
    fills = {"Tinggi": "F8CBAD", "Sedang": "FFE699", "Rendah": "C6EFCE"}; rc = cols["Risiko kebocoran (skrining otomatis)"]
    mc = cols["CEK MANUAL: urutan resampling vs split (full text)"]
    dv = DataValidation(type="list", formula1='"Bocor,Benar,Tidak jelas"', allow_blank=True); ws.add_data_validation(dv); dv.add(f"{mc}2:{mc}{ws.max_row}")
    for r in range(2, ws.max_row + 1):
        v = ws[f"{rc}{r}"].value
        if v in fills: ws[f"{rc}{r}"].fill = PatternFill("solid", fgColor=fills[v])
        for kname, col in cols.items():
            if kname.startswith("CEK MANUAL"): ws[f"{col}{r}"].fill = PatternFill("solid", fgColor="FFF2CC")
    for kname, wdt in {"Judul": 55, "Abstrak": 60, "Sumber": 30, "Model disebut": 28, "Alasan skrining": 35}.items(): ws.column_dimensions[cols[kname]].width = wdt
    ws.freeze_panes = "F2"; ws.auto_filter.ref = ws.dimensions; wb.save(xl); print("Tersimpan:", xl)
print("Semua hasil di:", OUT)